In [2]:
from datasets import load_dataset
  # ~860K math problems with chain-of-thought solutions
ds_cot = load_dataset('AI-MO/NuminaMath-CoT', cache_dir='datasets/numina-cot')
print(f'CoT: {len(ds_cot["train"])} train, {len(ds_cot["test"])} test')
print(ds_cot['train'][0].keys())  # See actual column names

# ~72K problems with tool-integrated reasoning (code execution)
ds_tir = load_dataset('AI-MO/NuminaMath-TIR', cache_dir='datasets/numina-tir')
print(f'TIR: {len(ds_tir["train"])} train, {len(ds_tir["test"])} test')
print(ds_tir['train'][0].keys())

# Validation benchmarks
ds_aime = load_dataset('AI-MO/aimo-validation-aime', cache_dir='datasets/val-aime')
ds_amc = load_dataset('AI-MO/aimo-validation-amc', cache_dir='datasets/val-amc')
print(f'AIME val: {len(ds_aime["train"])}')
print(f'AMC val: {len(ds_amc["train"])}')

/Users/tanzeel.shaikh/Library/Caches/pypoetry/virtualenvs/gnn-hgt-4314cFo1-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 100/100 [00:00<00:00, 35460.80 examples/s]


CoT: 859494 train, 100 test
dict_keys(['source', 'problem', 'solution', 'messages'])


Generating test split: 100%|██████████| 99/99 [00:00<00:00, 35972.98 examples/s]


TIR: 72441 train, 99 test
dict_keys(['problem', 'solution', 'messages'])


Generating train split: 100%|██████████| 83/83 [00:00<00:00, 25153.70 examples/s]

AIME val: 90
AMC val: 83


In [3]:
from datasets import load_dataset
from collections import Counter

ds = load_dataset('AI-MO/NuminaMath-CoT', cache_dir='datasets/numina-cot')

# NuminaMath has a 'source' field that tells you where each problem came from
sources = Counter(ds['train']['source'])
for source, count in sources.most_common(30):
      print(f"  {source}: {count}")

  cn_k12: 276554
  synthetic_math: 167874
  orca_math: 153314
  olympiads: 150563
  synthetic_amc: 62108
  aops_forum: 30192
  math: 7477
  gsm8k: 7342
  amc_aime: 4070


In [4]:
import json
from collections import Counter

with open('data/final/train/aopswiki.jsonl') as f:
    records = [json.loads(line) for line in f]

# Difficulty proxy: AIME problem number (P01=easy, P15=hard)
problem_numbers = []
for r in records:
    parts = r['id'].split('__')
    # e.g. aopswiki__AIME__2000__I__P01 → problem 1
    pnum = int(parts[-1].replace('P', ''))
    problem_numbers.append(pnum)

print(f"Total: {len(records)}")
print(f"Problem # distribution: {Counter(problem_numbers).most_common()}")
print(f"Answer range: min={min(r['final_answer'] for r in records)}, "
    f"max={max(r['final_answer'] for r in records)}")
print(f"Year range: {min(r['id'].split('__')[2] for r in records)} - "
    f"{max(r['id'].split('__')[2] for r in records)}")

# Split by difficulty tier
easy = [r for r in records if int(r['id'].split('__')[-1][1:]) <= 5]
medium = [r for r in records if 6 <= int(r['id'].split('__')[-1][1:]) <= 10]
hard = [r for r in records if int(r['id'].split('__')[-1][1:]) > 10]
print(f"\nEasy (P01-P05): {len(easy)}")
print(f"Medium (P06-P10): {len(medium)}")
print(f"Hard (P11-P15): {len(hard)}")

Total: 530
Problem # distribution: [(5, 43), (1, 42), (11, 41), (3, 37), (9, 36), (6, 36), (14, 35), (7, 34), (12, 34), (8, 34), (2, 34), (15, 32), (4, 32), (10, 31), (13, 29)]
Answer range: min=0, max=989
Year range: 2000 - 2025

Easy (P01-P05): 188
Medium (P06-P10): 171
Hard (P11-P15): 171


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from submission.aimo3_submission import Config, solve_problem

config = Config(
    model_id="Qwen/Qwen2.5-Math-7B-Instruct",
    num_samples=16,      # Fewer samples for speed during eval
    num_generations=4,
)

# --- Load model & tokenizer (MPS-compatible) ---
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.model_id)

print("Loading model...")
# Use float16 on MPS; if you hit OOM, switch to "auto" or use quantization
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    torch_dtype=torch.float16,
    device_map=device,           # "mps" on Apple Silicon, "auto" on CUDA
)
print(f"Model loaded on {device}")

# Load val-aime
ds = load_dataset('AI-MO/aimo-validation-aime', cache_dir='datasets/val-aime')

correct = 0
total = 0
for row in ds['train']:
    predicted = solve_problem(row['problem'], model, tokenizer, config)  # pass model & tokenizer
    gold = int(row['answer'])
    is_correct = (predicted == gold)
    correct += is_correct
    total += 1
    print(f"[{'OK' if is_correct else 'WRONG'}] predicted={predicted} gold={gold}")

print(f"\nBaseline: {correct}/{total} = {correct/total:.1%}")

Loading tokenizer...
Loading model...
